In [ ]:
from ibis import _
import pandas as pd

import src
from src.load import DataLoader

In [ ]:
pd.set_option("display.max_rows", 1024)
pd.set_option("display.max_colwidth", 256)

In [ ]:
data = DataLoader()

In [ ]:
videos = (
    data.videos(filtered=True)
    .join(data.channels().filter(_.channel != "FDP"), "channel_id")
    .select(
        ["video_id", "channel", "video_likes", "video_views", "video_uploadtime", "video_title"],
    )
    .to_pandas()
)

In [ ]:
quantiles = videos.groupby("channel").video_likes.quantile(q=0.99).rename("quantile")

In [ ]:
df = videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_likes > x["quantile"] else 0, axis=1)

In [ ]:
top_videos = (
    df.sort_values(["channel", "video_likes"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [ ]:
top_videos

In [ ]:
top_videos.columns

In [ ]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/top_videos_per_channel.csv", index=False,
)